<a href="https://colab.research.google.com/github/liuhuhuhuhu/financial-document-intelligence/blob/main/05_rag_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Financial Document Intelligence System

## 05. RAG Evaluation & Financial Analysis

**Dataset:** Apple FY2025 Form 10-K  
**Tools:** Python, Sentence Transformers, ChromaDB, Pandas

### Overview

This notebook evaluates the retrieval performance of
a financial document intelligence system and introduces
evidence-based financial analysis.

### Objectives

1. Rebuild the semantic retrieval pipeline.
2. Create a financial question-answering benchmark.
3. Assess retrieval relevance using manual annotations.
4. Calculate Hit@5 and Mean Reciprocal Rank (MRR@5).
5. Validate financial metrics and compute year-over-year growth.

### Evaluation Strategy

We use a manually reviewed set of questions and evidence
labels to assess the quality of retrieved financial disclosures.

### Pipeline

Financial Questions
    ↓
MiniLM Query Embeddings
    ↓
ChromaDB Retrieval
    ↓
Top-5 Evidence Chunks
    ↓
Human Relevance Labels
    ↓
Hit@5 / MRR@5


In [1]:
!pip -q install chromadb sentence-transformers pandas numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [4]:

import chromadb
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

print("All required libraries imported successfully!")
print("ChromaDB version:", chromadb.__version__)


All required libraries imported successfully!
ChromaDB version: 1.5.9


In [2]:

from google.colab import files

uploaded = files.upload()


Saving apple_2025_chunks.jsonl to apple_2025_chunks.jsonl


In [3]:

import json
import pandas as pd
import numpy as np

with open(
    "apple_2025_chunks.jsonl",
    "r",
    encoding="utf-8"
) as f:
    all_chunks = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Original chunks:", len(all_chunks))
print("First chunk:", all_chunks[0]["chunk_id"])
print("Example text:", all_chunks[0]["text"][:200])


Original chunks: 368
First chunk: apple_2025_p1_c1
Example text: UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒    ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal ye


In [5]:

filtered_chunks = [
    chunk for chunk in all_chunks
    if "TABLE OF CONTENTS" not in chunk["text"].upper()
]

texts = [
    chunk["text"]
    for chunk in filtered_chunks
]

print("Before filtering:", len(all_chunks))
print("After filtering:", len(filtered_chunks))
print("Removed:", len(all_chunks) - len(filtered_chunks))

assert len(filtered_chunks) > 0


Before filtering: 368
After filtering: 367
Removed: 1


In [6]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "Embedding dimension:",
    model.get_sentence_embedding_dimension()
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_1786/1238880213.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


In [7]:

chunk_embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Shape:", chunk_embeddings.shape)

assert chunk_embeddings.shape == (
    len(filtered_chunks), 384
)


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Shape: (367, 384)


In [10]:

import chromadb

client = chromadb.PersistentClient(
    path="./chroma_eval_data"
)

collection_name = "apple_2025_eval"

# Delete an existing collection only if it exists
existing_names = [
    c.name for c in client.list_collections()
]

if collection_name in existing_names:
    client.delete_collection(collection_name)
    print("Previous collection deleted.")

# Create a fresh collection
collection = client.create_collection(
    name=collection_name,
    configuration={
        "hnsw": {"space": "cosine"}
    }
)

print("Collection ready!")


Collection ready!


In [11]:

collection.add(
    ids=[c["chunk_id"] for c in filtered_chunks],
    documents=[c["text"] for c in filtered_chunks],
    embeddings=chunk_embeddings.tolist(),
    metadatas=[
        {
            "company": "Apple",
            "year": 2025,
            "page": int(c["page"])
        }
        for c in filtered_chunks
    ]
)

print("Stored chunks:", collection.count())

assert collection.count() == len(filtered_chunks)

print("Database quality check passed!")


Stored chunks: 367
Database quality check passed!


In [12]:

def retrieve_evidence(query, top_k=5):
    """Search for relevant financial document chunks."""

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    evidence = []

    for i in range(len(results["ids"][0])):
        evidence.append({
            "chunk_id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "page": results["metadatas"][0][i]["page"],
            "similarity": 1 - results["distances"][0][i]
        })

    return evidence


print("Retrieval function ready!")


Retrieval function ready!


In [13]:

test_query = "What are Apple's major risk factors?"

results = retrieve_evidence(
    test_query,
    top_k=5
)

for i, r in enumerate(results, start=1):
    print(
        f"Rank {i} | Page {r['page']} "
        f"| Similarity {r['similarity']:.4f}"
    )
    print(r["text"][:200])
    print("-" * 60)


Rank 1 | Page 9 | Similarity 0.6403
he Company’s suppliers are required to maintain safe working environments and operations, an industrial accident could occur 
and could result in serious injuries or loss of life, disruption to the Co
------------------------------------------------------------
Rank 2 | Page 11 | Similarity 0.6204
In addition, component suppliers may fail, 
be subject to consolidation within a particular industry, or decide to concentrate on the production of common components 
instead of components customized 
------------------------------------------------------------
Rank 3 | Page 8 | Similarity 0.6155
significant impact on the Company’s 
suppliers, contract manufacturers, logistics providers, distributors, cellular network carriers and other channel partners, and 
developers. Potential outcomes inc
------------------------------------------------------------
Rank 4 | Page 17 | Similarity 0.6107
ct the experience for users of the Company’s products and services, 

In [14]:

questions = [
    ("Q01", "What are Apple's major supply chain risks?"),
    ("Q02", "How could economic downturns affect Apple's operations?"),
    ("Q03", "What risks does Apple face from competition?"),
    ("Q04", "How could cybersecurity incidents affect Apple?"),
    ("Q05", "What were Apple's total net sales in FY2025?"),
    ("Q06", "How did total net sales change from FY2024 to FY2025?"),
    ("Q07", "What was Apple's net income in FY2025?"),
    ("Q08", "How much did Apple spend on R&D in FY2025?"),
    ("Q09", "Which product category generated the most net sales in FY2025?"),
    ("Q10", "What will Apple's stock price be in 2030?")
]

df_questions = pd.DataFrame(
    questions,
    columns=["question_id", "question"]
)

display(df_questions)


,question_id,question
0,Q01,What are Apple's major supply chain risks?
1,Q02,How could economic downturns affect Apple's op...
2,Q03,What risks does Apple face from competition?
3,Q04,How could cybersecurity incidents affect Apple?
4,Q05,What were Apple's total net sales in FY2025?
5,Q06,How did total net sales change from FY2024 to ...
6,Q07,What was Apple's net income in FY2025?
7,Q08,How much did Apple spend on R&D in FY2025?
8,Q09,Which product category generated the most net ...
9,Q10,What will Apple's stock price be in 2030?


In [15]:

retrieval_rows = []

for qid, question in questions:

    evidence = retrieve_evidence(
        question,
        top_k=5
    )

    for rank, item in enumerate(evidence, start=1):
        retrieval_rows.append({
            "question_id": qid,
            "question": question,
            "rank": rank,
            "chunk_id": item["chunk_id"],
            "page": item["page"],
            "similarity": item["similarity"],
            "text": item["text"]
        })

df_retrieval = pd.DataFrame(retrieval_rows)

print("Total retrieved rows:", len(df_retrieval))

display(
    df_retrieval[
        [
            "question_id",
            "rank",
            "page",
            "chunk_id",
            "similarity"
        ]
    ].head(15)
)


Total retrieved rows: 50


,question_id,rank,page,chunk_id,similarity
0,Q01,1,11,apple_2025_p11_c8,0.733302
1,Q01,2,9,apple_2025_p9_c8,0.696019
2,Q01,3,8,apple_2025_p8_c4,0.665234
3,Q01,4,17,apple_2025_p17_c8,0.608604
4,Q01,5,5,apple_2025_p5_c5,0.589881
5,Q02,1,8,apple_2025_p8_c4,0.661322
6,Q02,2,13,apple_2025_p13_c8,0.604068
7,Q02,3,24,apple_2025_p24_c3,0.598116
8,Q02,4,5,apple_2025_p5_c5,0.591371
9,Q02,5,30,apple_2025_p30_c5,0.582934


In [16]:

review = df_retrieval.copy()

review["relevant"] = ""
review["notes"] = ""

review.to_csv(
    "retrieval_review.csv",
    index=False,
    encoding="utf-8-sig"
)

from google.colab import files
files.download("retrieval_review.csv")

print("Review file exported!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Review file exported!


In [17]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

labels = pd.read_csv("retrieval_review_labeled.csv")

# Exclude the unanswerable question
labels = labels[labels["question_id"] != "Q10"].copy()

print("Labeled rows:", len(labels))
print("Relevant chunks:", labels["relevant"].sum())

Saving retrieval_review_labeled.csv to retrieval_review_labeled.csv
Labeled rows: 45
Relevant chunks: 18


In [18]:

# Q10 tests abstention, not evidence retrieval.
labels = labels[labels["question_id"] != "Q10"].copy()

labels["relevant"] = pd.to_numeric(
    labels["relevant"],
    errors="coerce"
)

if not labels["relevant"].isin([0, 1]).all():
    raise ValueError("All relevance labels must be 0 or 1.")

metrics = []

for qid, group in labels.groupby("question_id"):
    group = group.sort_values("rank")

    relevant_ranks = group.loc[
        group["relevant"] == 1, "rank"
    ].tolist()

    hit = int(len(relevant_ranks) > 0)

    reciprocal_rank = (
        1 / min(relevant_ranks)
        if relevant_ranks else 0
    )

    metrics.append({
        "question_id": qid,
        "hit_at_5": hit,
        "rr_at_5": reciprocal_rank
    })

df_metrics = pd.DataFrame(metrics)

print("Hit@5:", round(df_metrics["hit_at_5"].mean(), 3))
print("MRR@5:", round(df_metrics["rr_at_5"].mean(), 3))

display(df_metrics)


Hit@5: 0.889
MRR@5: 0.833


,question_id,hit_at_5,rr_at_5
0,Q01,1,1.0
1,Q02,1,1.0
2,Q03,1,1.0
3,Q04,1,1.0
4,Q05,1,1.0
5,Q06,1,0.5
6,Q07,1,1.0
7,Q08,0,0.0
8,Q09,1,1.0


In [19]:

query = (
    "Apple total net sales FY2025 FY2024 "
    "Consolidated Statements of Operations"
)

evidence = retrieve_evidence(query, top_k=10)

for item in evidence:
    print("Page:", item["page"])
    print("Chunk:", item["chunk_id"])
    print(item["text"][:600])
    print("-" * 60)


Page: 32
Chunk: apple_2025_p32_c1
Apple Inc.
CONSOLIDATED STATEMENTS OF OPERATIONS
(In millions, except number of shares, which are reflected in thousands, and per-share amounts)
Years ended
September 27,
2025
September 28,
2024
September 30,
2023
Net sales:
   Products
$ 
307,003 
$ 
294,866 
$ 
298,085 
   Services
 
109,158 
 
96,169 
 
85,200 
Total net sales
 
416,161 
 
391,035 
 
383,285 
Cost of sales:
   Products
 
194,116 
 
185,233 
 
189,282 
   Services
 
26,844 
 
25,119 
 
24,855 
Total cost of sales
 
220,960 
 
210,352 
 
214,137 
Gross margin
 
195,201 
 
180,683 
 
169,148 
Operating expenses:
Research and d
------------------------------------------------------------
Page: 36
Chunk: apple_2025_p36_c1
Apple Inc.
CONSOLIDATED STATEMENTS OF CASH FLOWS
(In millions)
Years ended
September 27,
2025
September 28,
2024
September 30,
2023
Cash, cash equivalents, and restricted cash and cash equivalents, beginning 
balances
$ 
29,943 
$ 
30,737 
$ 
24,977 
Operating activitie

In [21]:

# Verified Apple FY2025 Form 10-K figures
# Unit: USD millions

revenue_2025 = 416161
revenue_2024 = 391035

yoy_growth = (
    (revenue_2025 / revenue_2024) - 1
) * 100

print(f"FY2025 Revenue: ${revenue_2025:,.0f}M")
print(f"FY2024 Revenue: ${revenue_2024:,.0f}M")
print(f"Revenue YoY Growth: {yoy_growth:.2f}%")


FY2025 Revenue: $416,161M
FY2024 Revenue: $391,035M
Revenue YoY Growth: 6.43%



## 8. Financial Metric Retrieval Optimization

### 8.1 Error Analysis

The baseline MiniLM semantic retriever achieved:

- Hit@5: 0.889
- MRR@5: 0.833

However, Q08 (FY2025 R&D expenses) returned no
relevant evidence among the top five results.

### 8.2 Hypothesis

Dense semantic retrieval may underperform on
queries requiring exact financial metrics,
accounting terminology, and reporting periods.

We hypothesize that combining BM25 lexical search
with MiniLM semantic retrieval may improve
financial metric retrieval.

### 8.3 Experiment

We compare three retrieval methods:

1. MiniLM dense retrieval (baseline)
2. BM25 keyword retrieval
3. Hybrid retrieval using Reciprocal Rank Fusion

The comparison uses the same financial questions
and underlying document chunks.


In [22]:

import re

rd_candidates = []

for chunk in filtered_chunks:
    text = chunk["text"]

    if re.search(
        r"research\s+and\s+development|34,550",
        text,
        flags=re.IGNORECASE
    ):
        rd_candidates.append({
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "text": text
        })

print("R&D candidate chunks:", len(rd_candidates))

for item in rd_candidates[:12]:
    print("=" * 65)
    print("Chunk:", item["chunk_id"])
    print("PDF Page:", item["page"])
    print(item["text"][:1200])


R&D candidate chunks: 8
Chunk: apple_2025_p6_c6
PDF Page: 6
ead of components customized to meet the Company’s requirements, 
further limiting the Company’s ability to obtain sufficient quantities of components on commercially reasonable terms, or at all.
Research and Development
Because the industries in which the Company competes are characterized by rapid technological advances, the Company’s 
ability to compete successfully depends heavily upon its ability to ensure a continual and timely flow of competitive products, 
services and technologies to the marketplace. The Company continues to develop new technologies to enhance existing 
products and services, and to expand the range of its offerings through research and development (“R&D”), licensing of 
intellectual property and acquisition of third-party businesses and technology.
Intellectual Property
The Company currently holds a broad collection of intellectual property rights relating to certain aspects of its hardware, software

In [23]:

# Find chunks containing the exact FY2025 R&D figure

target = "34,550"

matches = [
    c for c in all_chunks
    if target in c["text"]
]

print("Exact amount matches:", len(matches))

for c in matches[:10]:
    print("=" * 60)
    print("Chunk:", c["chunk_id"])
    print("PDF page:", c["page"])
    print(c["text"][:1500])


Exact amount matches: 3
Chunk: apple_2025_p27_c2
PDF page: 27
vorable costs.
Services Gross Margin
Services gross margin increased during 2025 compared to 2024 primarily due to higher Services net sales and a different mix of 
services.
Services gross margin percentage increased during 2025 compared to 2024 primarily due to a different mix of services, partially 
offset by higher costs.
The Company’s future gross margins can be impacted by a variety of factors, as discussed in Part I, Item 1A of this Form 10-K 
under the heading “Risk Factors.” As a result, the Company believes, in general, gross margins will be subject to volatility and 
downward pressure.
Operating Expenses
Operating expenses for 2025, 2024 and 2023 were as follows (dollars in millions):
2025
Change
2024
Change
2023
Research and development
$ 
34,550 
 10 % $ 
31,370 
 5 % $ 
29,915 
Percentage of total net sales
 8% 
 8% 
 8% 
Selling, general and administrative
$ 
27,601 
 6 % $ 
26,097 
 5 % $ 
24,932 
Percentage 

In [24]:

import pandas as pd

summary = []

for c in matches:
    text = c["text"]
    pos = text.find("34,550")

    # Extract only the context surrounding the amount
    start = max(0, pos - 150)
    end = min(len(text), pos + 150)

    summary.append({
        "chunk_id": c["chunk_id"],
        "page": c["page"],
        "context": " ".join(
            text[start:end].split()
        )
    })

df_matches = pd.DataFrame(summary)

display(df_matches)


,chunk_id,page,context
0,apple_2025_p27_c2,27,"g Expenses Operating expenses for 2025, 2024 a..."
1,apple_2025_p32_c1,32,"55 Total cost of sales 220,960 210,352 214,137..."
2,apple_2025_p50_c3,50,"416,161 Cost of sales (95,699) (58,617) (35,14..."


In [25]:

!pip -q install rank-bm25


In [26]:

from rank_bm25 import BM25Okapi
import re
import numpy as np

def tokenize(text):
    return re.findall(
        r"[a-z0-9]+",
        text.lower()
    )

tokenized_corpus = [
    tokenize(chunk["text"])
    for chunk in filtered_chunks
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index ready!")
print("Indexed chunks:", len(tokenized_corpus))


BM25 index ready!
Indexed chunks: 367


In [27]:

def bm25_search(query, top_k=5):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = filtered_chunks[int(idx)]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(scores[idx]),
            "text": chunk["text"]
        })

    return results


In [28]:

query = "How much did Apple spend on R&D in FY2025?"

bm25_results = bm25_search(query, top_k=5)

for item in bm25_results:
    print(
        item["rank"],
        item["chunk_id"],
        "Page:", item["page"],
        "BM25:", round(item["score"], 3)
    )
    print(item["text"][:400])
    print("-" * 60)


1 apple_2025_p44_c3 Page: 44 BM25: 11.674
ptember  27, 2025, the Company had $4.7  billion in foreign tax credit carryforwards in Ireland and $4.0  billion in 
California R&D credit carryforwards, both of which can be carried forward indefinitely. A valuation allowance has been recorded 
for the credit carryforwards and a portion of other temporary differences.
Apple Inc. | 2025 Form 10-K | 41
------------------------------------------------------------
2 apple_2025_p11_c4 Page: 11 BM25: 11.602
n part by outsourcing partners located primarily 
in China mainland, India, Japan, South Korea, Taiwan and Vietnam, in addition to sourcing from partners and facilities located in 
the U.S. The Company relies on single-source partners in the U.S., Asia and Europe to supply and manufacture many 
components, and on partners primarily located in Asia, for final assembly of substantially all of the Co
------------------------------------------------------------
3 apple_2025_p1_c1 Page: 1 BM25: 10.87

In [29]:

def hybrid_search(query, top_k=5, candidate_k=30):
    # 1. Dense retrieval: MiniLM + ChromaDB
    dense_results = retrieve_evidence(
        query, top_k=candidate_k
    )

    # 2. Lexical retrieval: BM25
    lexical_results = bm25_search(
        query, top_k=candidate_k
    )

    # 3. Reciprocal Rank Fusion
    fused_scores = {}
    chunk_lookup = {
        c["chunk_id"]: c for c in filtered_chunks
    }

    for rank, item in enumerate(dense_results, start=1):
        cid = item["chunk_id"]
        fused_scores[cid] = (
            fused_scores.get(cid, 0) + 1 / (60 + rank)
        )

    for rank, item in enumerate(lexical_results, start=1):
        cid = item["chunk_id"]
        fused_scores[cid] = (
            fused_scores.get(cid, 0) + 1 / (60 + rank)
        )

    ranked_ids = sorted(
        fused_scores,
        key=fused_scores.get,
        reverse=True
    )[:top_k]

    return [
        {
            "rank": i,
            "chunk_id": cid,
            "page": chunk_lookup[cid]["page"],
            "text": chunk_lookup[cid]["text"],
            "rrf_score": fused_scores[cid]
        }
        for i, cid in enumerate(ranked_ids, start=1)
    ]


In [30]:

query = "How much did Apple spend on R&D in FY2025?"

dense = retrieve_evidence(query, top_k=5)
lexical = bm25_search(query, top_k=5)
hybrid = hybrid_search(query, top_k=5)

comparison = []

for name, results in [
    ("MiniLM", dense),
    ("BM25", lexical),
    ("Hybrid RRF", hybrid)
]:
    for rank, item in enumerate(results, start=1):
        comparison.append({
            "method": name,
            "rank": rank,
            "chunk_id": item["chunk_id"],
            "page": item["page"]
        })

import pandas as pd

df_comparison = pd.DataFrame(comparison)

display(df_comparison)


,method,rank,chunk_id,page
0,MiniLM,1,apple_2025_p27_c3,27
1,MiniLM,2,apple_2025_p30_c5,30
2,MiniLM,3,apple_2025_p24_c3,24
3,MiniLM,4,apple_2025_p5_c5,5
4,MiniLM,5,apple_2025_p26_c1,26
5,BM25,1,apple_2025_p44_c3,44
6,BM25,2,apple_2025_p11_c4,11
7,BM25,3,apple_2025_p1_c1,1
8,BM25,4,apple_2025_p21_c7,21
9,BM25,5,apple_2025_p27_c3,27


In [31]:

query = "How much did Apple spend on R&D in FY2025?"

target_id = "apple_2025_p27_c2"

methods = {
    "MiniLM": retrieve_evidence(
        query, top_k=collection.count()
    ),
    "BM25": bm25_search(
        query, top_k=len(filtered_chunks)
    ),
    "Hybrid RRF": hybrid_search(
        query,
        top_k=len(filtered_chunks),
        candidate_k=len(filtered_chunks)
    )
}

for method, results in methods.items():
    target_rank = next(
        (
            i for i, item in enumerate(results, start=1)
            if item["chunk_id"] == target_id
        ),
        None
    )

    print(f"{method}: Target chunk rank = {target_rank}")


MiniLM: Target chunk rank = 105
BM25: Target chunk rank = 294
Hybrid RRF: Target chunk rank = 189


In [32]:

original_query = "How much did Apple spend on R&D in FY2025?"

expanded_query = (
    "Apple research and development "
    "operating expenses fiscal year 2025"
)

target_id = "apple_2025_p27_c2"

for query_name, query in [
    ("Original Query", original_query),
    ("Expanded Query", expanded_query)
]:
    print(f"\n{'=' * 60}")
    print(query_name)
    print("Query:", query)

    for method_name, search_fn in [
        ("MiniLM", lambda q: retrieve_evidence(q, top_k=5)),
        ("BM25", lambda q: bm25_search(q, top_k=5)),
        ("Hybrid", lambda q: hybrid_search(q, top_k=5))
    ]:
        results = search_fn(query)

        ids = [r["chunk_id"] for r in results]

        target_rank = (
            ids.index(target_id) + 1
            if target_id in ids
            else None
        )

        print(
            f"{method_name}: "
            f"Target in Top 5 = {target_rank is not None}, "
            f"Rank = {target_rank}"
        )



Original Query
Query: How much did Apple spend on R&D in FY2025?
MiniLM: Target in Top 5 = False, Rank = None
BM25: Target in Top 5 = False, Rank = None
Hybrid: Target in Top 5 = False, Rank = None

Expanded Query
Query: Apple research and development operating expenses fiscal year 2025
MiniLM: Target in Top 5 = False, Rank = None
BM25: Target in Top 5 = True, Rank = 2
Hybrid: Target in Top 5 = False, Rank = None


In [33]:

import re

def normalize_financial_query(query):
    """Expand common financial terminology."""

    normalized = query.lower()

    replacements = {
        r"\br&d\b": "research and development",
        r"\bfy(\d{4})\b": r"fiscal year \1",
        r"\byoy\b": "year over year",
        r"\bnet sales\b": "total net sales",
    }

    for pattern, replacement in replacements.items():
        normalized = re.sub(
            pattern,
            replacement,
            normalized,
            flags=re.IGNORECASE
        )

    return normalized

query = "How much did Apple spend on R&D in FY2025?"

print("Original:", query)
print("Normalized:", normalize_financial_query(query))


Original: How much did Apple spend on R&D in FY2025?
Normalized: how much did apple spend on research and development in fiscal year 2025?


In [34]:

original_query = "How much did Apple spend on R&D in FY2025?"
normalized_query = normalize_financial_query(original_query)

results = bm25_search(normalized_query, top_k=5)

target_id = "apple_2025_p27_c2"

for result in results:
    is_target = result["chunk_id"] == target_id

    print(
        f"Rank {result['rank']}: "
        f"{result['chunk_id']} "
        f"{'✓ TARGET' if is_target else ''}"
    )


Rank 1: apple_2025_p37_c2 
Rank 2: apple_2025_p24_c2 
Rank 3: apple_2025_p37_c1 
Rank 4: apple_2025_p24_c1 
Rank 5: apple_2025_p4_c3 


In [35]:

import re

def normalize_financial_query_v2(query):
    """Normalize financial questions for lexical retrieval."""

    q = query.lower()

    # Expand financial abbreviations
    q = re.sub(
        r"\br\s*&\s*d\b",
        "research and development",
        q
    )

    q = re.sub(
        r"\bfy\s*(\d{4})\b",
        r"fiscal year \1",
        q
    )

    # Map common expressions to accounting terminology
    if "research and development" in q:
        q = re.sub(
            r"\b(spend|spent|spending)\b",
            "operating expenses",
            q
        )

    # Remove common question words
    q = re.sub(
        r"\b(how|much|did|in|on|the|what|was)\b",
        " ",
        q
    )

    return " ".join(q.split())


original_query = "How much did Apple spend on R&D in FY2025?"

normalized_v2 = normalize_financial_query_v2(original_query)

print("Original:", original_query)
print("Normalized V2:", normalized_v2)


Original: How much did Apple spend on R&D in FY2025?
Normalized V2: apple operating expenses research and development fiscal year 2025?


In [36]:

target_id = "apple_2025_p27_c2"

results = bm25_search(normalized_v2, top_k=5)

for r in results:
    marker = "✓ TARGET" if r["chunk_id"] == target_id else ""
    print(
        f"Rank {r['rank']}: "
        f"{r['chunk_id']} | Page {r['page']} {marker}"
    )


Rank 1: apple_2025_p27_c3 | Page 27 
Rank 2: apple_2025_p27_c2 | Page 27 ✓ TARGET
Rank 3: apple_2025_p32_c1 | Page 32 
Rank 4: apple_2025_p37_c2 | Page 37 
Rank 5: apple_2025_p50_c4 | Page 50 


In [37]:

comparison_rows = []

for qid, question in questions:
    if qid == "Q10":
        continue

    for method, query in [
        ("BM25 Original", question),
        ("BM25 Normalized V2",
         normalize_financial_query_v2(question))
    ]:
        results = bm25_search(query, top_k=5)

        comparison_rows.append({
            "question_id": qid,
            "method": method,
            "top_1_chunk": results[0]["chunk_id"],
            "top_5_chunks": ", ".join(
                r["chunk_id"] for r in results
            )
        })

df_query_comparison = pd.DataFrame(comparison_rows)

display(df_query_comparison)


,question_id,method,top_1_chunk,top_5_chunks
0,Q01,BM25 Original,apple_2025_p9_c8,"apple_2025_p9_c8, apple_2025_p25_c2, apple_202..."
1,Q01,BM25 Normalized V2,apple_2025_p9_c8,"apple_2025_p9_c8, apple_2025_p11_c5, apple_202..."
2,Q02,BM25 Original,apple_2025_p8_c2,"apple_2025_p8_c2, apple_2025_p16_c4, apple_202..."
3,Q02,BM25 Normalized V2,apple_2025_p8_c2,"apple_2025_p8_c2, apple_2025_p13_c8, apple_202..."
4,Q03,BM25 Original,apple_2025_p6_c2,"apple_2025_p6_c2, apple_2025_p10_c5, apple_202..."
5,Q03,BM25 Normalized V2,apple_2025_p6_c2,"apple_2025_p6_c2, apple_2025_p10_c5, apple_202..."
6,Q04,BM25 Original,apple_2025_p14_c3,"apple_2025_p14_c3, apple_2025_p20_c5, apple_20..."
7,Q04,BM25 Normalized V2,apple_2025_p14_c3,"apple_2025_p14_c3, apple_2025_p20_c5, apple_20..."
8,Q05,BM25 Original,apple_2025_p27_c2,"apple_2025_p27_c2, apple_2025_p39_c1, apple_20..."
9,Q05,BM25 Normalized V2,apple_2025_p25_c4,"apple_2025_p25_c4, apple_2025_p24_c2, apple_20..."



## Key Findings & Conclusions

### Baseline Evaluation
- Hit@5: 0.889
- MRR@5: 0.833
- Evaluation set: 9 answerable financial questions

### Financial Retrieval Error Analysis
- Q08 failed to retrieve the FY2025 R&D expense
  in the baseline Top-5 results.
- The correct evidence was identified in
  `apple_2025_p27_c2`.
- BM25 with financial query normalization V2
  retrieved the target chunk at Rank 2.
- The initial Hybrid RRF approach did not
  resolve the Q08 retrieval failure.

### Limitations
- Small manually labeled evaluation dataset.
- Query normalization was developed using Q08.
- Results do not establish general improvement
  across unseen financial questions.
- Financial table structure remains imperfect.

### Future Work
- Expand the evaluation benchmark.
- Implement table-aware financial retrieval.
- Evaluate financial QA numerical accuracy.
- Deploy the RAG pipeline as an interactive application.
